In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.init as init
from torch.autograd import Variable

In [14]:
class LambdaLayer(nn.Module):
    def __init__(self, lambd):
        super(LambdaLayer, self).__init__()
        self.lambd = lambd

    def forward(self, x):
        return self.lambd(x)

In [ ]:

class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, option='A'):
        super(BasicBlock, self).__init__()
        self.conv1r = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=True)
        init.kaiming_normal_(self.conv1r.weight)
        self.conv1g = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=True)
        self.conv1g.weight.data.fill_(0)
        self.conv1g.bias.data.fill_(1)
        self.conv1b = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=True)
        self.conv1b.weight.data.fill_(0)
        self.conv1b.bias.data.fill_(0)      
  
        self.bn1 = nn.BatchNorm2d(planes)
        
        self.conv2r = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=True)
        init.kaiming_normal_(self.conv2r.weight)
        self.conv2g = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=True)
        self.conv2g.weight.data.fill_(0)
        self.conv2g.bias.data.fill_(1)
        self.conv2b = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=True)
        self.conv2b.weight.data.fill_(0)
        self.conv2b.bias.data.fill_(0)      
  
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            if option == 'A':
                """
                For CIFAR10 ResNet paper uses option A.
                """
                self.shortcut = LambdaLayer(lambda x:
                                            F.pad(x[:, :, ::2, ::2], (0, 0, 0, 0, planes//4, planes//4), "constant", 0))
            elif option == 'B':
                self.shortcut = nn.Sequential(
                     nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False),
                     nn.BatchNorm2d(self.expansion * planes)
                )

    def forward(self, x):
        if self.ReLU:
            out = F.relu(self.bn1(self.conv1r(x)*self.conv1g(x)+self.conv1b(x.pow(2))))
            out = self.bn2(self.conv2r(out)*self.conv2g(out)+self.conv2b(out.pow(2)))
            out += self.shortcut(x)
            out = F.relu(out)
        else:
            out = self.bn1(self.conv1r(x)*self.conv1g(x)+self.conv1b(x.pow(2)))
            out = self.bn2(self.conv2r(out)*self.conv2g(out)+self.conv2b(out.pow(2)))
            out += self.shortcut(x)
            out = out
        return out


In [ ]:

class QResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10, ReLU=True):
        super(QResNet, self).__init__()
        self.in_planes = 16
        self.ReLU = ReLU        # Changes if the model is run with our without ReLU

        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1, bias=False) # for grey and white
        # self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False) # for RGB
        self.bn1 = nn.BatchNorm2d(16)
        self.layer1 = self._make_layer(block, 16, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 32, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 64, num_blocks[2], stride=2)
        self.linear = nn.Linear(64, num_classes)



    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion

        return nn.Sequential(*layers)

    def forward(self, x):
        if self.ReLU:
            out = F.relu(self.bn1(self.conv1(x)))
            out = self.layer1(out)
            out = self.layer2(out)
            out = self.layer3(out)
            out = F.avg_pool2d(out, out.size()[3])
            out = out.view(out.size(0), -1)
            out = self.linear(out)
        else:
            out = self.bn1(self.conv1(x))
            out = self.layer1(out)
            out = self.layer2(out)
            out = self.layer3(out)
            out = F.avg_pool2d(out, out.size()[3])
            out = out.view(out.size(0), -1)
            out = self.linear(out)
        
        return out


In [17]:
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader

# Load MNIST CSV: first column = label, remaining columns = pixel intensities
df = pd.read_csv("mnist_train.csv", header=None)

y = torch.tensor(df.iloc[:, 0].values, dtype=torch.long)
X = torch.tensor(df.iloc[:, 1:].values, dtype=torch.float32) / 255.0  # normalize to [0, 1]

In [18]:
# Infer image size (expects 28x28 for MNIST)
num_pixels = X.shape[1]
side = int(num_pixels ** 0.5)
assert side * side == num_pixels, f"Expected square image, got {num_pixels} pixels."
X = X.view(-1, 1, side, side)  # NCHW for Conv2d: [batch, channel, height, width]

In [19]:


def qresnet20():
    return QResNet(BasicBlock, [1, 1, 1])


def qresnet32():
    return QResNet(BasicBlock, [5, 5, 5])


def qresnet44():
    return QResNet(BasicBlock, [7, 7, 7])


def qresnet56():
    return QResNet(BasicBlock, [9, 9, 9])


def qresnet110():
    return QResNet(BasicBlock, [18, 18, 18])


def qresnet1202():
    return QResNet(BasicBlock, [200, 200, 200])


In [20]:

train_loader = DataLoader(TensorDataset(X, y), batch_size=128, shuffle=True)

# Build model from existing net_name (e.g., 'qresnet1202')
model = globals()["qresnet20"]()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Train
epochs = 5
for epoch in range(epochs):
    print(f"Starting epoch {epoch}")
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Loss: {running_loss/total:.4f} | "
        f"Acc: {correct/total:.4f}"
    )

cuda
Starting epoch 0
Epoch 1/5 | Loss: 0.1811 | Acc: 0.9607
Starting epoch 1
Epoch 2/5 | Loss: 0.0367 | Acc: 0.9905
Starting epoch 2
Epoch 3/5 | Loss: 0.0247 | Acc: 0.9931
Starting epoch 3
Epoch 4/5 | Loss: 0.0181 | Acc: 0.9950
Starting epoch 4
Epoch 5/5 | Loss: 0.0149 | Acc: 0.9956


In [21]:


def test(net):
    import numpy as np
    total_params = 0

    for x in filter(lambda p: p.requires_grad, net.parameters()):
        total_params += np.prod(X.data.numpy().shape)
    print("Total number of params", total_params)
    print("Total layers", len(list(filter(lambda p: p.requires_grad and len(p.data.size())>1, net.parameters()))))



In [ ]:
__all__ = ['ResNet', 'qresnet20', 'qresnet32', 'qresnet44', 'qresnet56', 'qresnet110', 'qresnet1202']

In [25]:
__all__ = ['qresnet20']
for net_name in __all__:
    if net_name.startswith('qresnet'):
        print(net_name)
        test(globals()[net_name]())
        print()

qresnet20
Total number of params 2493120000
Total layers 20



In [26]:
# Load test data
df_test = pd.read_csv("mnist_test.csv", header=None)
y_test = torch.tensor(df_test.iloc[:, 0].values, dtype=torch.long)
X_test = torch.tensor(df_test.iloc[:, 1:].values, dtype=torch.float32) / 255.0
X_test = X_test.view(-1, 1, 28, 28)

# Create test loader
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=128, shuffle=False)

# Evaluate
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f"Test Accuracy: {correct/total:.4f} ({correct}/{total})")

Test Accuracy: 0.9930 (9930/10000)
